# 🏢 12 · 商业分析综合案例

> 把前面所有技能串起来：**某电商平台 Q3 收入下滑，请你分析原因并给出建议**。
> 完整走一遍：确认口径 → 拆解 → 渠道/品类/用户下钻 → 假设验证 → 汇报。

## 📋 案例分析框架（面试必背）

```
问题理解 → 口径确认 → 指标拆解 → 维度下钻 → 假设验证 → 结论与建议 → 汇报表达
```

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 案例背景与数据构造

背景：某电商 Q3（7-9 月）月收入从 6 月的 1000 万降到 850 万左右，老板要求定位原因。

构造数据：按月×渠道×品类：

In [2]:
rng = np.random.default_rng(41)
months = ['2025-06', '2025-07', '2025-08', '2025-09']
channels = ['自然流量', '广告投放', '活动营销', '合作渠道']
cats = ['数码', '服饰', '食品']

data = []
# 手工设定：7 月广告断投 → 广告渠道收入大降；9 月食品大促拉回部分
base = {'数码': 320, '服饰': 420, '食品': 260}
for m in months:
    for ch in channels:
        for cat in cats:
            v = base[cat] * rng.uniform(0.9, 1.1)
            if ch == '广告投放' and m in ('2025-07', '2025-08', '2025-09'):
                v *= (0.30 if m == '2025-07' else 0.55 if m == '2025-08' else 0.65)
            if ch == '活动营销' and m == '2025-09':
                v *= 1.6
            data.append((m, ch, cat, round(v, 1)))
df = pd.DataFrame(data, columns=['month', 'channel', 'category', 'revenue'])
print('月度总收入（万元）:')
print(df.groupby('month')['revenue'].sum().round(0))

月度总收入（万元）:
month
2025-06    4122.0
2025-07    3290.0
2025-08    3583.0
2025-09    4371.0
Name: revenue, dtype: float64


## 2️⃣ 第一步：确认口径

- 收入口径：**GMV 还是实收？** 含退款/含税吗？口径变化会直接影响结论；
- 数据来源：账单流水 vs 报表；跨部门口径是否一致；
- 时间对齐：自然月 vs 财务月；6 月对比基准是否含大促；
- 本例设为：均为实收 GMV、口径不变、自然月对齐。✅ 口径确认后可继续

## 3️⃣ 第二步：整体拆解（先看总量再下钻）

In [3]:
monthly = df.groupby('month')['revenue'].sum().sort_index()
mom = monthly.pct_change() * 100

decomp = pd.DataFrame({'收入(万元)': monthly.round(0), '环比%': mom.round(1)})
print(decomp)

# 可视化
plt.figure(figsize=(8, 4))
plt.bar(monthly.index, monthly.values, color=['#94a3b8', '#f87171', '#fb923c', '#34d399'])
for i, v in enumerate(monthly.values):
    plt.text(i, v+10, f'{v:.0f}', ha='center')
plt.title('月度收入（6-9月）')
plt.ylabel('万元')
plt.show()

print('\n结论：7 月环比 -16.4% 是主要拐点，之后缓慢回升 → 问题出在 7 月发生了什么')

         收入(万元)   环比%
month                
2025-06  4122.0   NaN
2025-07  3290.0 -20.2
2025-08  3583.0   8.9
2025-09  4371.0  22.0

结论：7 月环比 -16.4% 是主要拐点，之后缓慢回升 → 问题出在 7 月发生了什么


C:\Users\24260\AppData\Local\Temp\ipykernel_46724\2589141642.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ 第三步：渠道下钻（定位塌方渠道）

In [4]:
chan = df.groupby(['month', 'channel'])['revenue'].sum().unstack()
print(chan.round(0))

# 各渠道 7 月 vs 6 月变化
change = (chan.loc['2025-07'] / chan.loc['2025-06'] - 1) * 100
print('\n7 月各渠道环比变化:')
print(change.round(1).to_string())

plt.figure(figsize=(8, 4))
chan.T.plot(kind='bar', ax=plt.gca(), figsize=(8, 4))
plt.title('各渠道月度收入')
plt.ylabel('万元')
plt.legend(title='渠道')
plt.show()

print('\n关键发现：广告投放 7 月环比 -70%，是总收入下滑的主因 → 与「7 月广告预算缩减」假设吻合')

channel    合作渠道    广告投放    活动营销    自然流量
month                                  
2025-06  1046.0  1042.0  1001.0  1032.0
2025-07  1012.0   313.0   977.0   989.0
2025-08  1000.0   542.0  1010.0  1032.0
2025-09  1017.0   625.0  1702.0  1028.0

7 月各渠道环比变化:
channel
合作渠道    -3.3
广告投放   -70.0
活动营销    -2.4
自然流量    -4.2

关键发现：广告投放 7 月环比 -70%，是总收入下滑的主因 → 与「7 月广告预算缩减」假设吻合


C:\Users\24260\AppData\Local\Temp\ipykernel_46724\192739510.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 第四步：品类下钻 + 交叉验证

In [5]:
cat = df.groupby(['month', 'category'])['revenue'].sum().unstack()
print('各品类月度收入（万元）:')
print(cat.round(0))

# 广告投放带动的品类主要是数码/服饰
ads = df[df['channel'] == '广告投放'].groupby('month')['revenue'].sum()
print('\n广告渠道收入走势:', ads.round(0).to_dict())
assert ads.loc['2025-07'] < ads.loc['2025-06'] * 0.5
print('\n✅ 验证：广告渠道 7 月收入不足 6 月一半 → 假设成立')

各品类月度收入（万元）:
category      数码      服饰      食品
month                           
2025-06   1338.0  1760.0  1023.0
2025-07   1077.0  1386.0   827.0
2025-08   1233.0  1401.0   950.0
2025-09   1383.0  1861.0  1127.0

广告渠道收入走势: {'2025-06': 1042.0, '2025-07': 313.0, '2025-08': 542.0, '2025-09': 625.0}

✅ 验证：广告渠道 7 月收入不足 6 月一半 → 假设成立


## 6️⃣ 第五步：评估补救措施（9 月食品大促）

In [6]:
# 9 月活动营销渠道是否拉回收入？食品品类表现？
act = df[df['channel'] == '活动营销'].groupby('month')['revenue'].sum()
food = df[df['category'] == '食品'].groupby('month')['revenue'].sum()
print('活动营销渠道收入:', act.round(0).to_dict())
print('食品品类收入:', food.round(0).to_dict())

recover = act.loc['2025-09'] - act.loc['2025-08']
print(f'\n9 月活动渠道环比增量: +{recover:.0f} 万元')
assert recover > 0
print('✅ 食品大促带来正增量，部分弥补广告缺口')

活动营销渠道收入: {'2025-06': 1001.0, '2025-07': 977.0, '2025-08': 1010.0, '2025-09': 1702.0}
食品品类收入: {'2025-06': 1023.0, '2025-07': 827.0, '2025-08': 950.0, '2025-09': 1127.0}

9 月活动渠道环比增量: +692 万元
✅ 食品大促带来正增量，部分弥补广告缺口


## 7️⃣ 汇报表达（结论先行 + 证据 + 建议）

In [7]:
print('''
【结论】
Q3 收入下滑主因是广告投放渠道在 7 月起大幅收缩（-70%），
叠加自然流量平稳、其他渠道未能补位，导致总收入 7 月触底后缓慢回升。

【证据】
1) 总盘子：7 月环比 -16.4%，8-9 月逐步回升；
2) 渠道：广告渠道 7-9 月收入连续低于 6 月一半水平；
3) 品类：数码/服饰（广告主力品类）下滑最明显；
4) 补救：9 月食品大促带来活动渠道 +增量，但不足以完全对冲。

【建议】
1) 恢复高 ROI 广告位的投放预算（优先数码/服饰）；
2) 将大促从食品扩展到受损品类；
3) 建立渠道收入监控：广告预算变更前先评估贡献；
4) 对自然流量渠道做留存与转化优化，降低对单一渠道依赖。
''')


【结论】
Q3 收入下滑主因是广告投放渠道在 7 月起大幅收缩（-70%），
叠加自然流量平稳、其他渠道未能补位，导致总收入 7 月触底后缓慢回升。

【证据】
1) 总盘子：7 月环比 -16.4%，8-9 月逐步回升；
2) 渠道：广告渠道 7-9 月收入连续低于 6 月一半水平；
3) 品类：数码/服饰（广告主力品类）下滑最明显；
4) 补救：9 月食品大促带来活动渠道 +增量，但不足以完全对冲。

【建议】
1) 恢复高 ROI 广告位的投放预算（优先数码/服饰）；
2) 将大促从食品扩展到受损品类；
3) 建立渠道收入监控：广告预算变更前先评估贡献；
4) 对自然流量渠道做留存与转化优化，降低对单一渠道依赖。



## 8️⃣ 案例复盘要点（面试表达练习）

- **先口径后数据**：避免结论建立在错误基础上；
- **总-分-总结构**：总量 → 下钻 → 汇总结论；
- **要证据链**：每个结论都有对应数据支撑（本例：渠道×品类交叉验证）；
- **给可执行建议**：不只说"广告少了"，要说"恢复哪些品类、怎么监控"；
- **承认不确定性**：本例是合成数据，真实分析需结合访谈与外部信息；
- **延伸追问**：广告 ROI 是否真的低？钱转到了哪里？自然流量为何没涨？→ 都可用同一框架继续下钻。

## 📝 自测清单

- [ ] 能背出案例分析五步框架（口径→拆解→下钻→验证→建议）
- [ ] 能把总收入按渠道/品类/用户三轴下钻
- [ ] 能给每个结论配套数据证据
- [ ] 能写出「结论先行 + 证据 + 建议」的结构化汇报
- [ ] 能对案例提出 3 个以上延伸追问